# Chapitre 2 — Extraction des données (sources locales)


---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Extraire** des données depuis différents formats de fichiers (CSV, Excel, JSON) en utilisant les paramètres appropriés de pandas
2. **Connecter** Python à une base de données SQL et exécuter des requêtes pour récupérer des données dans un DataFrame
3. **Consommer** des APIs REST en gérant l'authentification, la pagination et les limites de requêtes
4. **Évaluer** quand le web scraping est approprié et appliquer les principes éthiques et légaux

---

## 2.4 Introduction au web scraping

### ⚠️ Avertissement important

Le web scraping est un outil puissant mais **juridiquement sensible**. Avant de scraper :

1. ✅ Vérifiez s'il existe une **API officielle**
2. ✅ Lisez les **CGU (Conditions Générales d'Utilisation)**
3. ✅ Respectez le fichier **robots.txt**
4. ✅ N'extrayez **jamais de données personnelles** sans base légale

### Le cadre légal français (CNIL, 2025)

La CNIL a publié en juin 2025 des directives claires :

> « Le traitement ne pourra pas entrer dans les attentes raisonnables des personnes si son responsable n'exclut pas de la collecte les sites qui s'opposent clairement au moissonnage par l'intermédiaire des protocoles d'exclusion robots.txt ou de CAPTCHA. »

**Amendes possibles :** Jusqu'à **20 millions d'euros** ou 4% du chiffre d'affaires mondial (RGPD).

**Exemple réel :** KASPR a été condamné à **240 000 €** pour avoir scrapé LinkedIn sans consentement.

*(Source : [CNIL - Web Scraping Guidance 2025](https://www.cnil.fr/en/legal-basis-legitimate-interests-focus-sheet-measures-implement-case-data-collection-web-scraping))*

---

### Vérifier robots.txt

Avant tout scraping, consultez `robots.txt` à la racine du site :

In [ ]:

# Vérifier le robots.txt d'un site de test
robots_url = "https://www.data.gouv.fr/robots.txt"
response = requests.get(robots_url)
# Afficher le contenu
print("=" * 50)
print(f"CONTENU DE {robots_url}")
print("=" * 50)
print(response.text)

**Interprétation d'un robots.txt :**
```
User-agent: *
Disallow: /admin/
Disallow: /private/
Crawl-delay: 10

User-agent: GPTBot
Disallow: /
```

- Tous les bots (`*`) peuvent accéder au site sauf `/admin/` et `/private/`
- Ils doivent attendre **10 secondes** entre les requêtes
- GPTBot (OpenAI) est **totalement bloqué**

In [ ]:
# Vérifier le robots.txt d'un site de test
robots_url = "https://www.wikipedia.org/robots.txt"
response = requests.get(robots_url)
# Afficher le contenu
print("=" * 50)
print(f"CONTENU DE {robots_url}")
print("=" * 50)
print(response.text)

In [ ]:
# Wikipedia a des exigences spécifiques
headers = {
    'User-Agent': 'MonBotEducatif/'
}

robots_url = "https://www.wikipedia.org/robots.txt"
response = requests.get(robots_url, headers=headers, timeout=10)

# Afficher le contenu
print("=" * 50)
print(f"CONTENU DE {robots_url}")
print("=" * 50)
print(response.text)


---

**Explication détaillée du message robots.txt de Wikipedia**

 **Structure générale :**

**1. En-tête et avertissements**
```
robots.txt for http://www.wikipedia.org/ and friends
```
→ Ce fichier s'applique à wikipedia.org ET à tous les sous-domaines/sites amis (wikimedia, etc.)

```
Please note: There are a lot of pages on this site, (...) your access to the site may be blocked.
```
→ **Message important** : 
- Wikipedia a ÉNORMÉMENT de pages
- Certains robots/scrapers vont TROP VITE
- **Si vous n'êtes pas responsable, vous pouvez être BLOQUÉ**

---

**2. Exemples de règles spécifiques**

**A. Blocage de robots problématiques :**
```robots
# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# http://mj12bot.com/
User-agent: MJ12bot
Disallow: /
```
**Explication :**
- `MJ12bot` est un robot qui a été observé en train de :
  1. Spammer les pages avec `?curid=NNNNNN`
  2. Ignorer les réponses HTTP 429 ("Trop de requêtes")
  3. Pourtant, il prétend respecter robots.txt
- `Disallow: /` = **BLOQUÉ COMPLÈTEMENT** (interdit sur TOUTE la site)

**B. Blocage des bots publicitaires :**
```robots
# advertising-related bots:
User-agent: Mediapartners-Google*
Disallow: /
```
**Explication :**
- Tous les user-agents qui commencent par `Mediapartners-Google` sont bloqués
- Ce sont des robots liés à la publicité (Google AdSense)
- Wikipedia étant sans publicité, elle les bloque

**C. Robots autorisés (bots de travail Wikipedia) :**
```robots
# Wikipedia work bots:
User-agent: IsraBot
Disallow:

User-agent: Orthogaffe
...
```
**Explication :**
- `IsraBot` et `Orthogaffe` sont des **bots officiels de Wikipedia**
- `Disallow:` **vide** = **TOUT EST AUTORISÉ** pour ces bots
- Ce sont des robots qui aident à la maintenance du site

---

**3. Règles pour tous les robots (`*`)**

(La suite de votre fichier contient probablement :) 
```robots
User-agent: *
Disallow: /w/
Disallow: /api/
Disallow: /trap/
...
```

**Signification :**
- `User-agent: *` = **s'applique à TOUS les robots**
- Les chemins comme `/w/`, `/api/`, `/trap/` sont interdits
- Mais l'accès aux articles (`/wiki/`) est généralement autorisé

---

#### **Points pédagogiques importants :**

##### **1. La philosophie de Wikipedia :**
- **"Nous sommes ouverts mais pas naïfs"**
- Les contenus sont libres (Creative Commons)
- Mais ils protègent leurs serveurs contre les abus

##### **2. Les types de règles :**
- **Blocage total** : `Disallow: /` (pour les mauvais robots)
- **Permissions totales** : `Disallow:` (vide, pour leurs bots)
- **Restrictions partielles** : chemins spécifiques interdits

##### **3. Le ton du fichier :**
- **Pédagogique** : ils expliquent POURQUOI ils bloquent
- **Transparent** : ils citent les URLs des robots problématiques
- **Avertissement clair** : "si vous abusez, vous serez bloqués"

##### **4. Ce que cela nous apprend sur le web scraping :**

**FAIRE :**
- ✅ Mettre un User-Agent clair et descriptif
- ✅ Respecter les `Disallow`
- ✅ Suivre les politiques de rate limiting
- ✅ Lire les commentaires (#) qui expliquent les règles

**NE PAS FAIRE :**
- ❌ Ignorer les codes HTTP 429 (trop de requêtes)
- ❌ Faire du spam
- ❌ Prétendre être un autre robot
- ❌ Aller trop vite

---

##### **En résumé :**

Ce robots.txt montre que Wikipedia :
1. **Veut partager la connaissance** (accès ouvert aux articles)
2. **Doit se protéger** (blocage des abuseurs)
3. **Éduque les webmasters** (commentaires explicatifs)
4. **Fait des distinctions** (pas tous les robots sont traités pareil)

**Leçon clé :** robots.txt n'est pas juste une liste technique, c'est une **politique d'accès** qui reflète les valeurs du site.


---

### BeautifulSoup : les bases

In [ ]:
# Installation (exécuter une seule fois si nécessaire)
!pip3 install beautifulsoup4

In [ ]:
"""
DÉMONSTRATION DE WEB SCRAPING RESPONSABLE
Site utilisé : https://quotes.toscrape.com (site conçu pour l'apprentissage)

IMPORTANT : 
- Ce site est fait pour s'entraîner au scraping
- Sur un vrai site, TOUJOURS vérifier robots.txt et conditions d'utilisation
- Respecter le rate limiting (ne pas surcharger les serveurs)
"""

import requests
from bs4 import BeautifulSoup
import time  # Pour le rate limiting

#### ÉTAPE 1 : VÉRIFICATION ROBOTS.TXT (BONNE PRATIQUE ESSENTIELLE)

In [ ]:
print("ÉTAPE 1 : Vérification de robots.txt")

# URL du site
BASE_URL = "https://quotes.toscrape.com"
# "https://www.data.gouv.fr" 

# 1. Lire robots.txt pour voir ce qui est autorisé
try:
    robots_response = requests.get(f"{BASE_URL}/robots.txt", headers=headers, timeout=5)
    if robots_response.status_code == 200:
        print("✅ robots.txt trouvé. Extrait :")
        print("-" * 40)
        print(robots_response.text[:300] + "..." if len(robots_response.text) > 300 else robots_response.text)
    else:
        print("ℹ️  Pas de robots.txt ou inaccessible")
except:
    print("⚠️  Impossible d'accéder à robots.txt (peut être normal)")

#### ÉTAPE 2 : CONFIGURATION RESPONSABLE

In [ ]:
print("ÉTAPE 2 : Configuration responsable")


"""
POURQUOI ces headers ?
- User-Agent : identifie votre bot (transparence)
- Accept-Language : préférence linguistique
- Accept : format de réponse accepté
- From : email de contact (bonne pratique)
"""
headers = {
    'User-Agent': 'MonBotApprentissage/1.0 (formation scraping; contact@example.com)',
    'Accept-Language': 'fr-FR,fr;q=0.9',
    'Accept': 'text/html,application/xhtml+xml',
    'From': 'contact@example.com'  # Email de contact (bonne pratique)
}

print(f"📱 User-Agent utilisé : {headers['User-Agent']}")
print("   → Permet au site de savoir qui fait la requête")
print("   → Important pour la transparence et le debugging")

time.sleep(1)  # Pause d'1 seconde par politesse (rate limiting)

#### ÉTAPE 3 : RÉCUPÉRATION DE LA PAGE

In [ ]:
print("ÉTAPE 3 : Récupération de la page HTML")


print(f"🌐 Envoi de la requête vers : {BASE_URL}")

try:
    # Envoi de la requête HTTP GET avec nos headers
    response = requests.get(BASE_URL, headers=headers, timeout=10)
    
    # Vérification du statut HTTP
    print(f"📊 Statut HTTP reçu : {response.status_code}")
    
    if response.status_code == 200:
        print("✅ Succès ! Page récupérée.")
        print(f"📄 Taille du HTML : {len(response.content)} caractères")
    else:
        print(f"❌ Erreur : Code {response.status_code}")
        exit()  # On arrête si la page n'est pas accessible
        
except requests.exceptions.Timeout:
    print("❌ Timeout : le serveur ne répond pas")
    exit()
except Exception as e:
    print(f"❌ Erreur inattendue : {e}")
    exit()

#### ÉTAPE 4 : PARSING DU HTML AVEC BEAUTIFSOUP

In [ ]:
print("ÉTAPE 4 : Parsing du HTML avec BeautifulSoup")

"""
QU'EST-CE QUE LE PARSING ?
C'est l'analyse de la structure HTML pour pouvoir
naviguer et extraire des données facilement.
"""

# Création de l'objet BeautifulSoup
# 'html.parser' est le parser intégré à Python
soup = BeautifulSoup(response.content, 'html.parser')

print("✅ HTML parsé avec succès !")
print(f"   Type de l'objet soup : {type(soup)}")
print(f"   Balise racine : {soup.find().name}")

#### ÉTAPE 5 : EXTRACTION DE DONNÉES SIMPLES

In [ ]:
print("ÉTAPE 5 : Extraction de données")

# -------------------------------------------------
# 5.1 : Extraire le titre de la page
# -------------------------------------------------

print("\n 1 Extraction du titre :")

titre = soup.find("title")  # Trouve la première balise <title>

if titre:
    print(f"   Balise <title> trouvée : '{titre.text}'")
    print(f"   Longueur : {len(titre.text)} caractères")
else:
    print("   ❌ Aucun titre trouvé")


In [ ]:
# -------------------------------------------------
# 5.2 : Compter les citations
# -------------------------------------------------

print("\n 2 Recherche des citations :")

# Trouver TOUTES les balises <span> avec la classe "text"

quotes = soup.find_all("span", class_="text")
# Alternative : soup.select("span.text") avec CSS selectors

print(f"   Nombre total de citations trouvées : {len(quotes)}")

# Afficher les 3 premières citations
if quotes:
    print(f"   Les 3 premières citations :")
    for i, quote in enumerate(quotes[:3], 1):
        # .text extrait le texte sans les balises HTML
        # [:80] limite à 80 caractères pour l'affichage
        citation = quote.text.strip()
        print(f"   {i}. {citation[:80]}{'...' if len(citation) > 80 else ''}")
else:
    print("   ❌ Aucune citation trouvée")

In [ ]:
# -------------------------------------------------
# 5.3 : Extraire les auteurs
# -------------------------------------------------

print("\n 3 Recherche des auteurs :")

auteurs = soup.find_all("small", class_="author")

if auteurs:
    print(f"   Nombre d'auteurs trouvés : {len(auteurs)}")
    print(f"   Les 3 premiers auteurs :")
    for i, auteur in enumerate(auteurs[:3], 1):
        print(f"   {i}. {auteur.text}")
else:
    print("   ❌ Aucun auteur trouvé")


### Sélecteurs courants

| Méthode | Usage | Exemple |
|---------|-------|--------|
| `find()` | Premier élément | `soup.find("h1")` |
| `find_all()` | Tous les éléments | `soup.find_all("p")` |
| `select()` | Sélecteur CSS | `soup.select("div.content > p")` |
| `.text` | Texte de l'élément | `element.text` |
| `.get()` | Attribut | `link.get("href")` |

---

### 🔄 Point de vue alternatif : Scraping vs APIs vs Datasets publics

> **Alternative Viewpoint** : Avant de scraper, posez-vous ces questions :
> 1. **L'API existe-t-elle ?** La plupart des grands sites ont des APIs officielles (Twitter, Reddit, etc.)
> 2. **Le dataset existe-t-il ?** Kaggle, data.gouv.fr, et autres plateformes offrent des données déjà extraites
> 3. **Puis-je demander ?** Contacter directement l'entreprise peut aboutir à un accès officiel
>
> Le scraping devrait être le **dernier recours**, pas le premier réflexe.

### Selenium

C'est l'outil qui va te permettre de passer du "Scraping de base" au **"Scraping tout-terrain"**.

Si `requests` est comme un sniper qui récupère un fichier précis, **Selenium** est un **pilote de ligne** : il prend les commandes d'un vrai navigateur (Chrome, Firefox, Edge) et le dirige exactement comme un être humain le ferait.

---

#### 1. Pourquoi en as-tu besoin (Le problème du JS) ?

Comme tu l'as vu avec YouTube ou Amazon, beaucoup de sites modernes sont "vides" quand on récupère leur code source brut. Ils ont besoin de **JavaScript** pour construire la page (afficher les vidéos, les prix, etc.).

* **`requests`** : Télécharge le code, mais ne peut pas exécuter le JavaScript. Il voit la page "avant" que les éléments ne soient dessinés.
* **`Selenium`** : Ouvre le navigateur, attend que le JavaScript s'exécute, et **attend que les éléments apparaissent** avant de les scrapper.

---

#### 2. Ce que Selenium peut faire (et que BeautifulSoup ne peut pas)

Avec Selenium, tu peux coder des actions interactives :

1. **Cliquer** sur un bouton "Accepter les cookies".
2. **Scroller** vers le bas de la page pour charger plus de vidéos (le fameux "Infinite Scroll" de YouTube).
3. **Saisir du texte** dans une barre de recherche et appuyer sur "Entrée".
4. **Prendre une capture d'écran** de la page pour prouver que le scrap s'est bien passé.

---

#### 3. Comparaison : Sniper vs Bulldozer

Voici un tableau comparatif pour ton cours :

| Caractéristique | Requests + BeautifulSoup | Selenium |
| --- | --- | --- |
| **Vitesse** | Très rapide (Sniper) | Lent (doit charger tout le navigateur) |
| **JavaScript** | ❌ Impossible | ✅ Parfaitement géré |
| **Interaction** | ❌ Aucune | ✅ Clics, Scroll, Saisie |
| **Détection** | Facile à bloquer | Plus difficile (on ressemble à un humain) |
| **Ressources** | Très léger | Gourmand en RAM/CPU |

[Image comparing requests vs Selenium for web scraping]

---

#### 4. À quoi ressemble le code Selenium ?

Pour l'utiliser, il faut installer la bibliothèque et un petit programme appelé "Driver" (qui fait le pont entre Python et ton navigateur).

```python
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

# 1. On lance le navigateur (Chrome ici)
driver = webdriver.Chrome()

# 2. On va sur YouTube
driver.get("https://www.youtube.com")

# 3. On attend un peu que le JS charge les vidéos
time.sleep(5)

# 4. On récupère les titres (exactement comme avec BeautifulSoup, mais via le driver)
titres = driver.find_elements(By.ID, "video-title")

for titre in titres[:5]:
    print(titre.text)

# 5. On ferme le navigateur
driver.quit()

```

---

#### 5. Le concept du "Headless" (Pour les pros)

Une fois que ton script est prêt, tu n'as pas forcément envie de voir la fenêtre Chrome s'ouvrir et bouger toute seule sur ton écran (surtout sur un serveur).
On active alors le mode **"Headless"** (sans tête) : le navigateur travaille en arrière-plan, de manière invisible, mais il exécute toujours le JavaScript.

---

#### Conseil 

Ne commencez **jamais** par Selenium si `requests` suffit. Selenium est puissant mais il est "lourd" et casse plus souvent à cause des mises à jour des navigateurs.
